# 9.1 Q-테이블의 한계와 함수근사 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter09_1_qnet_generalization.ipynb)

책 본문: [Chapter 9.1](https://smhanlab.com/book-ml/kor/ml2/chapter09.html)

**1차원 랜덤워크 장난감 환경**의 동일한 300개 에피소드(시드 7)에
**Q-테이블**(0.1칸, 61칸)과 **신경망**(1→16→16→1)을 각각 학습시켜
두 근사기의 차이를 확인합니다: "본 칸만 정확"(보간 없음) vs.
"보지 못한 곳까지 보간"(일반화).

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

## 1. 장난감 환경: 1차원 랜덤워크

- 위치 \(x \in [0,6]\), 시작 위치는 [0.2, 5.8] 균일 무작위
- 각 스텝 ±0.6 (동등 확률)
- 0에 닿으면 에피소드 종료(터미널 보상 \(-1\)), 6에 닿으면 종료(터미널 보상 \(+1\)), 중간 보상은 0
- 할인율 \(\gamma = 0.9\)

300개 에피소드(시드 7)를 생성하고, 각 에피소드의 각 상태에 대한 리턴 \(G_t\)를 계산합니다. 이 데이터가 **테이블과 신경망이 함께 쓰는 유일한 데이터**입니다.

In [2]:
import numpy as np

GAMMA = 0.9
STEP = 0.6
N_EPISODES = 300
SEED = 7
NBINS = 61

rng = np.random.default_rng(SEED)

def rollout(rng):
    """한 에피소드: [0.2, 5.8] 균일 무작위에서 시작, ±STEP 동등 확률.
    0/6 경계에 닿으면 종료(터미널 위치는 정확히 0.0 / 6.0으로 기록)."""
    x = rng.uniform(0.2, 5.8)
    xs = [x]
    while True:
        x = x + STEP if rng.random() < 0.5 else x - STEP
        if x >= 6.0:
            xs.append(6.0)
            break
        if x <= 0.0:
            xs.append(0.0)
            break
        xs.append(x)
    return xs

# 300개 에피소드 (시드 7) -- 테이블과 신경망이 공유하는 동일한 데이터
episodes = [rollout(rng) for _ in range(N_EPISODES)]
n_steps = sum(len(e) - 1 for e in episodes)
print(f"{N_EPISODES}개 에피소드 (시드 {SEED}), 총 {n_steps} 스텝, "
      f"에피소드당 평균 {n_steps / N_EPISODES:.1f} 스텝")

# 각 상태의 리턴 G_t: 중간 r=0, 터미널 r=±1, G_t = sum_{k>=t} GAMMA^{k-t} r_k
all_states, all_returns = [], []
for xs in episodes:
    T = len(xs) - 1
    term = 1.0 if xs[-1] >= 6.0 else -1.0
    Gt = np.zeros(T + 1)
    Gt[T] = term
    for t in range(T - 1, -1, -1):
        Gt[t] = GAMMA * Gt[t + 1]
    all_states.extend(xs)
    all_returns.extend(Gt)

S = np.asarray(all_states)
G = np.asarray(all_returns)
print(f"(상태, 리턴) 쌍: {len(S)}개 (터미널 상태의 리턴 = ±1)")

300개 에피소드 (시드 7), 총 6481 스텝, 에피소드당 평균 21.6 스텝
(상태, 리턴) 쌍: 6781개 (터미널 상태의 리턴 = ±1)


In [3]:
# 참값 V*(x): 균등 정책(0.5/0.5)의 벨만 방정식 고정점
#   V(x) = 0.5*GAMMA*[V(x+STEP) 또는 +1] + 0.5*GAMMA*[V(x-STEP) 또는 -1]
# 경계 V(0)=-1, V(6)=+1을 고정하고 값 반복(value iteration)으로 푼다.
grid = np.linspace(0.0, 6.0, 601)
V = np.zeros_like(grid)
V[0], V[-1] = -1.0, 1.0
d = np.inf
for it in range(1, 20001):
    Vr = np.interp(grid + STEP, grid, V)
    Vl = np.interp(grid - STEP, grid, V)
    Vr[grid + STEP >= 6.0] = 1.0
    Vl[grid - STEP <= 0.0] = -1.0
    V_new = 0.5 * GAMMA * (Vr + Vl)
    V_new[0], V_new[-1] = -1.0, 1.0
    d = float(np.max(np.abs(V_new - V)))
    V = V_new
    if d < 1e-10:
        break
print(f"V* 계산 완료: {it}회 반복 (최종 최대 변화 {d:.1e})")
print(f"V*(0)={V[0]:.3f}  V*(3)={V[300]:.3f}  V*(6)={V[-1]:.3f}")

V* 계산 완료: 137회 반복 (최종 최대 변화 9.9e-11)
V*(0)=-1.000  V*(3)=-0.000  V*(6)=1.000


## 2. Q-테이블 (0.1칸, 61칸)

각 칸에 그 칸을 방문한 상태들의 **평균 리턴**을 저장합니다(리턴이 곧 MC 목표). 방문한 적 없는 칸은 아예 답이 없습니다.

In [4]:
# 테이블: 각 칸(0.1칸, 61칸)에 그 칸을 방문한 상태들의 평균 리턴
bin_of = np.clip(np.round(S / 0.1).astype(np.int64), 0, NBINS - 1)
tab = np.zeros(NBINS)
cnt = np.zeros(NBINS)
np.add.at(tab, bin_of, G)
np.add.at(cnt, bin_of, 1)
tab[cnt > 0] /= cnt[cnt > 0]
print(f"방문한 칸: {int((cnt > 0).sum())}/{NBINS}")

방문한 칸: 61/61


## 3. 신경망 (1→16→16→1, MSE)

테이블과 **동일한 (상태, 리턴) 데이터**를 **연속 함수**로 학습합니다. 입력은 [0,1]로 정규화합니다.

In [5]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)
net = nn.Sequential(
    nn.Linear(1, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
lossf = nn.MSELoss()

X = torch.tensor(S / 6.0, dtype=torch.float32).unsqueeze(1)   # (n,1), [0,1] 정규화
Y = torch.tensor(G, dtype=torch.float32).unsqueeze(1)         # (n,1)

for epoch in range(1, 301):
    opt.zero_grad()
    loss = lossf(net(X), Y)
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  MSE {loss.item():.5f}")

epoch  50  MSE 0.08898
epoch 100  MSE 0.08887
epoch 150  MSE 0.08887
epoch 200  MSE 0.08887
epoch 250  MSE 0.08887


epoch 300  MSE 0.08887


## 4. 그림: 표 vs 신경망 vs 참값 V*

(a) 전 구간 — 표는 칸마다 점프(보간 없음), 신경망은 매끄러운 곡선.
(b) 목표(6) 근처 확대 — 표의 "칸"이 참값의 급격한 변화를 놓치고, 신경망이 그 사이를 보간합니다.

In [6]:
IMG = "/home/smhan/book-ml/kor/src/images"

xs_grid = np.linspace(0.0, 6.0, 601)
net_v = net(torch.tensor(xs_grid / 6.0, dtype=torch.float32).unsqueeze(1)).detach().numpy().ravel()
bin_centers = np.arange(NBINS) * 0.1

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.bar(bin_centers, tab, width=0.1, color="0.75", zorder=2, label="Q-table (0.1 bin, mean return)")
ax.plot(xs_grid, V, "k-", lw=1.8, zorder=3, label="True V*")
ax.plot(xs_grid, net_v, color="tab:blue", lw=1.5, zorder=3, label="Neural net (1→16→16→1)")
ax.axhline(0, color="0.8", lw=0.8)
ax.set_xlabel("Position x")
ax.set_ylabel("Value")
ax.set_title("(a) Full range")
ax.legend(fontsize=9)

ax = axes[1]
ax.bar(bin_centers, tab, width=0.1, color="0.75", zorder=2, label="Q-table")
ax.plot(xs_grid, V, "k-", lw=1.8, zorder=3, label="True V*")
ax.plot(xs_grid, net_v, color="tab:blue", lw=1.5, zorder=3, label="Neural net")
ax.axhline(0, color="0.8", lw=0.8)
ax.set_xlim(4.5, 6.0)
ax.set_xlabel("Position x")
ax.set_ylabel("Value")
ax.set_title("(b) Zoom near the goal (6)")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(IMG + "/ch09_1_table_vs_net.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch09_1_table_vs_net.svg")

저장: /home/smhan/book-ml/kor/src/images/ch09_1_table_vs_net.svg


## 5. "본 적 없는 상태"에 대한 답

테이블은 방문한 적 없는 칸에 대해 답을 낼 수 없지만, 신경망은 어떤 입력에도(주변에서 배운 것으로) 보간된 답을 냅니다.

In [7]:
# 학습 중 한 번도 등장하지 않는 위치를 골라 세 근사기의 답을 비교
x_new = 3.0123
b = int(np.clip(np.round(x_new / 0.1), 0, NBINS - 1))
print(f"x = {x_new} (학습에서 한 번도 만난 적 없는 위치)")
print(f"  참값 V*  = {np.interp(x_new, grid, V):.4f}")
print(f"  신경망   = {net(torch.tensor([[x_new / 6.0]], dtype=torch.float32)).item():.4f}  (주변을 보고 보간)")
if cnt[b] > 0:
    print(f"  Q-테이블 = {tab[b]:.4f}  (칸 {b}의 평균 리턴 -- '칸 내부' 어디든 같은 값)")
else:
    print(f"  Q-테이블 = 칸 {b} 방문 없음 -> 아예 답이 없음")

x = 3.0123 (학습에서 한 번도 만난 적 없는 위치)
  참값 V*  = 0.0363
  신경망   = -0.0047  (주변을 보고 보간)
  Q-테이블 = 0.0153  (칸 30의 평균 리턴 -- '칸 내부' 어디든 같은 값)


In [8]:
import math

# 본문 "Q-함수를 신경망으로 근사": QNetwork(4, 2)의 파라미터 수
qnet = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
print(f"QNetwork(4, 2) 파라미터 수: {sum(p.numel() for p in qnet.parameters()):,}")

# 본문 "Q-테이블이 감당 못 하는 규모": Atari 84x84 흑백 화면의 상태 수
print(f"Atari 84x84 화면 상태 공간: 256^7056 ≈ 10^{7056 * math.log10(256):,.0f}")

QNetwork(4, 2) 파라미터 수: 4,610
Atari 84x84 화면 상태 공간: 256^7056 ≈ 10^16,993


## 핵심 정리

- **표는 본 칸에서만 정확하다** — 칸과 칸 사이에는 보간이 없고, 방문한 적 없는 칸에는 답이 없다.
- **신경망은 연속 함수다** — "비슷한 상태 → 비슷한 값"을 배워 본 적 없는 상태까지 보간한다. 이것이 **일반화**이고, 함수근사를 쓰는 진짜 이유다.
- 둘은 **동일한** 300개 에피소드 데이터(시드 7)로 학습했다 — 차이의 원인은 데이터가 아니라 근사기다.